In [27]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv(override=True)

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)


In [28]:
student_db={}


In [29]:
# tool 1
def add_marks(name,marks):
    student_db[name] = marks
    return {"status":"saved"}

In [30]:
#tool 2
def show_marks():
    return student_db

In [31]:
#step 2 Tool schemas 

add_marks_json={
    "name":"add_marks",
    "description":"Add student marks",
    "parameters":{
        "type":"object",
        "properties":{
            "name":{"type":"string"},
            "marks":{"type":"integer"}
        },
     "required":["name","marks"]
    }
}

In [32]:
show_marks_json = {
    "name":"show_marks",
    "description":"Show marks of student",
    "parameters":{
        "type":"object",
        "properties":{}
    }
}

In [33]:
tools = [
    {"type": "function", "function": add_marks_json},
    {"type": "function", "function": show_marks_json}
]

In [34]:
available_tools={
    "add_marks":add_marks,
    "show_marks":show_marks
}

In [35]:
from ast import arguments
import json

def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(
            tool_call.function.arguments
        )

        tool = available_tools.get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({
            "role":"tool",
            "content": json.dumps(result),
            "tool_call_id":tool_call.id
        })

    return results

In [36]:
#Agent loop 
def chat(user_message):
    message = [
        {"role":"user","content":user_message}
    ]
    done = False
    while not done:
        response = client.chat.completions.create(
                    model="gemini-2.5-flash",
                    messages=message,
                    tools=tools
                )
        finish_reason = response.choices[0].finish_reason
        print(finish_reason)
        if finish_reason == "tool_calls":
            assistant_message = response.choices[0].message
            tool_calls  = assistant_message.tool_calls
            tool_result = handle_tool_calls(tool_calls)
            message.append(assistant_message)
            message.extend(tool_result)
        else:
            done = True
    
    return response.choices[0].message.content

In [37]:
chat("Add marks for kedar = 85")

tool_calls
stop


'Successfully added marks for Kedar.'

In [38]:
chat("show marks")

tool_calls
stop


'Kedar: 85'